### MY Digital Twin

In [ ]:
# imports
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
import smtplib
import traceback
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from pypdf import PdfReader
import gradio as gr
from IPython.display import Markdown, display
from bs4 import BeautifulSoup   

In [160]:
# The usual start
load_dotenv(override=True)

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
model = "google/gemini-2.0-flash-001"

open_router = os.environ.get("OPENROUTER_API_KEY")
google = os.environ.get("GOOGLE_API_KEY")

# Global client for OpenRouter
client = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=open_router
)

In [161]:
response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": "Tell a joke"}],
    stream=True
)

full_response = ""
display_handle = display(Markdown(""), display_id=True)

for chunk in response:
    content = chunk.choices[0].delta.content
    if content:
        full_response += content
        display_handle.update(Markdown(full_response))

Why don't scientists trust atoms?

Because they make up everything!


In [162]:
# For Phone pushover Message config set-up 
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user not found
Pushover token not found


In [163]:
# For Mail configuration set-up
email_user = os.getenv("EMAIL")
email_password = os.getenv("PASSWORD")

if email_user:
    print(f"Email user found and starts with {email_user[0]}")
else:
    print("Email user not found")

if email_password:
    print("Email password found")
else:
    print("Email password not found")

Email user found and starts with m
Email password found


In [164]:
# Email sending utility
def send_mail(subject, body, recipient=email_user):
    try:    
        msg = MIMEMultipart()
        msg['From'] = f"Omnicom Solutions <{email_user}>"
        msg['To'] = recipient
        msg['Subject'] = subject
        msg.attach(MIMEText(body, 'plain'))
        
        # SSL (port 465) is much more reliable for Gmail
        print(f"Connecting to SMTP SSL for {recipient}...")
        server = smtplib.SMTP_SSL('smtp.gmail.com', 465, timeout=12)
        server.login(email_user, email_password)
        server.send_message(msg)
        server.quit()
        print("Email sent successfully")
        return True
    except Exception as e:
        print(f"Error sending email: {e}")
        return False

In [165]:
# This calls the push over message 
def push(message):
    print(f"Push: {message}")
    if pushover_user and pushover_token:
        payload = {"user": pushover_user, "token": pushover_token, "message": message}
        requests.post(pushover_url, data=payload)

In [166]:
# This records users' details for possible feedback. It is going to be a tool for the LLM
def record_user_details(email, name="Name not provided", notes="not provided"):
    msg = f"Recording interest from {name} with email {email} and notes {notes}"
    print(msg)
    push(msg)
    # Directly triggering email notification for reliability
    send_mail(f"New Interest from Website: {name}", f"User Email: {email}\nNotes: {notes}")
    return {"recorded": "ok", "notification": "sent"}

In [167]:
# This record questions the LLM can't answer and want human to answer. To become a tool for the LLM
def record_unknown_question(question, contact_info):
    msg = f"Recording question asked that I couldn't answer: {question}. User contact: {contact_info}"
    print(msg)
    push(msg)
    # Directly triggering email notification for reliability
    send_mail(
        "Unknown Question Recorded on Website", 
        f"A question was asked that the agent couldn't answer:\n\nQuestion: {question}\n\nVisitor Contact Details provided:\n{contact_info}"
    )
    return {"recorded": "ok", "notification": "sent"}

In [168]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address. This will also trigger an email notification to me.",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            },
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [169]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered, BUT ONLY AFTER the user has provided their contact information.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The specific question the user asked that you couldn't answer."
            },
            "contact_info": {
                "type": "string",
                "description": "The email address or phone number the user explicitly provided so Mustapha can get back to them. DO NOT call this tool without asking the user for this first."
            }
        },
        "required": ["question", "contact_info"],
        "additionalProperties": False
    }
}

In [170]:
send_mail_json = {
    "name": "send_mail",
    "description": "Use this tool to send an email for any other purpose not covered by recording details or questions.",
    "parameters": {
        "type": "object",
        "properties": {
            "subject": {
                "type": "string",
                "description": "The subject of the email"
            },
            "body": {
                "type": "string",
                "description": "The body content of the email"
            },
            "recipient": {
                "type": "string",
                "description": "The recipient's email address. Defaults to the administrator's email if not provided."
            }
        },
        "required": ["subject", "body"],
        "additionalProperties": False
    }
}

In [171]:
tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json},
    {"type": "function", "function": send_mail_json}
]

In [172]:
# Define your digital presence URLs here
my_urls = [
    "https://github.com/Rabiu-Adeola-Mustapha", # Replace with your actual GitHub profile
    "https://www.linkedin.com/in/mustapha-rabiu-adeola-4aa689192/" # Replace with your actual LinkedIn link
]

def scrape_urls(urls):
    combined_text = ""
    # Realistic browser headers to reduce bot-blocking on sites like GitHub
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Accept-Language': 'en-US,en;q=0.9',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Referer': 'https://www.google.com/'
    }
    for url in urls:
        try:
            print(f"Scraping {url}...")
            response = requests.get(url, headers=headers, timeout=12)
            response.raise_for_status() 
            soup = BeautifulSoup(response.content, 'html.parser')
            # Extract visible text
            text = soup.get_text(separator=' ', strip=True)
            combined_text += f"\n--- Content from {url} ---\n{text[:5000]}" # Limit length to avoid blowing up context
        except Exception as e:
            print(f"Warning: Failed to scrape {url}: {e}")
    return combined_text

# IMPORTANT: If automatic scraping fails (common for social media like LinkedIn),
# paste your bio, CV or LinkedIn summary here manually as a string below.
fallback_profile_data = """
Enter your bio or LinkedIn details here manually if the scraper gets blocked!
"""

scraped_data = scrape_urls(my_urls)
combined_raw_data = scraped_data + "\n--- Manual Fallback Data ---\n" + fallback_profile_data

Scraping https://github.com/Rabiu-Adeola-Mustapha...
Scraping https://www.linkedin.com/in/mustapha-rabiu-adeola-4aa689192/...


In [173]:
name = "Mustapha Rabiu Adeola"

print("Generating digital presence summary from the web...")
summary_prompt = f"Please read the following scraped and fallback web data about {name} and create two things:\n" \
                 f"1. A professional summary of his background and skills.\n" \
                 f"2. A structured representation resembling a LinkedIn profile.\n\n" \
                 f"Data Context:\n{combined_raw_data}"

summary_response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": summary_prompt}]
)

generated_context = summary_response.choices[0].message.content
print("Generated Context Successfully!")

# Assign to variables used by the system prompt
summary = generated_context
linkedin = "Included in the summary above."

Generating digital presence summary from the web...
Generated Context Successfully!


In [174]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible.\n\n" \
f"## STRICT GUIDELINES:\n" \
f"1. **Personal Facts**: ONLY use the provided Summary and LinkedIn profile information for facts about {name}. If the information is not present, do not make it up or use general knowledge. Admit you don't know.\n" \
f"2. **Source Attribution**: When providing an answer about {name}, mention your source where appropriate (e.g., 'According to my GitHub...', 'Based on my LinkedIn profile...', or 'From my professional summary...').\n" \
f"3. **Unknown Question Protocol**: If you can't answer a question, DO NOT immediately call the `record_unknown_question` tool. FIRST, tell the user you don't know, and ask them for their email or phone number so Mustapha can get back to them. ONLY call `record_unknown_question` AFTER they have explicitly provided their contact information.\n" \
f"4. **Tone**: Professional, engaging, and faithfully representing {name}.\n" \
f"5. **General Topics**: You may use general knowledge to explain technical terms or concepts mentioned in {name}'s profile (e.g., explaining what AWS is if mentioned), but keep personal career facts restricted to the context below.\n\n" \
f"## Summary Content:\n{summary}\n\n" \
f"## LinkedIn Profile Content:\n{linkedin}\n\n" \
f"With this context, please chat with the user, always staying in character as {name}."

In [175]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        try:
            function_name = tool_call.function.name
            kwargs = json.loads(tool_call.function.arguments)
            
            print(f"Executing tool: {function_name} with args: {kwargs}")
            
            if function_name == "record_user_details":
                output = record_user_details(**kwargs)
            elif function_name == "record_unknown_question":
                output = record_unknown_question(**kwargs)
            elif function_name == "send_mail":
                success = send_mail(kwargs.get('subject', 'No Subject'), kwargs.get('body', ''), kwargs.get('recipient', email_user))
                output = {"status": "email sent" if success else "email failed"}
            else:
                output = {"error": f"Unknown function {function_name}"}
                
            results.append({"role": "tool", "tool_call_id": tool_call.id, "name": function_name, "content": json.dumps(output)})
        except Exception as e:
            print(f"Error executing tool {tool_call.function.name}: {e}")
            results.append({"role": "tool", "tool_call_id": tool_call.id, "name": tool_call.function.name, "content": json.dumps({"error": str(e)})})
    return results

def chat(message, history):
    # Standardize messages for the LLM call
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    
    done = False
    max_iterations = 5
    iterations = 0
    
    while not done and iterations < max_iterations:
        iterations += 1
        try:
            response = client.chat.completions.create(model=model, messages=messages, tools=tools)
            choice = response.choices[0]
            finish_reason = choice.finish_reason
            response_message = choice.message
            
            if finish_reason == "tool_calls" and response_message.tool_calls:
                tool_calls = response_message.tool_calls
                tool_results = handle_tool_calls(tool_calls)
                
                # Append assistant tool call request and the results
                messages.append(response_message.model_dump(exclude_unset=True))
                messages.extend(tool_results)
            else:
                done = True
        except Exception as e:
            print(f"LLM Chat Error:\n{traceback.format_exc()}")
            return f"Oops! Something went wrong: {str(e)}"
            
    content = response.choices[0].message.content
    
    # Make sure we don't accidentally leak prompt instructions if it outputs something weird
    if not content or "Make sure" in content or "Please provide" in content:
        content = "I have successfully recorded your inquiry and notified Mustapha so he can review it personally. Is there anything else I can assist you with today?"
            
    return content


In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


Executing tool: record_unknown_question with args: {'contact_info': 'ask@gmail.com', 'question': 'Do you have Fin-Tech experience ?'}
Recording question asked that I couldn't answer: Do you have Fin-Tech experience ?. User contact: ask@gmail.com
Push: Recording question asked that I couldn't answer: Do you have Fin-Tech experience ?. User contact: ask@gmail.com
Connecting to SMTP SSL for mustopharabiu@gmail.com...
Email sent successfully
